In [35]:
import pandas as pd
import numpy as np

recipe = pd.read_json('./train.json')
recipe

,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."
...,...,...,...
39769,29109,irish,"[light brown sugar, granulated sugar, butter, ..."
39770,11462,italian,"[KRAFT Zesty Italian Dressing, purple onion, b..."
39771,2238,irish,"[eggs, citrus fruit, raisins, sourdough starte..."
39772,41882,chinese,"[boneless chicken skinless thigh, minced garli..."


In [39]:
all_ingredients = []
for indiv_ingredient_list in recipe['ingredients'].values:
    for ingredient in indiv_ingredient_list:
        all_ingredients.append(ingredient)
ingredients_together = pd.DataFrame(all_ingredients)

In [24]:
unique_ingredients = ingredients_together.squeeze().value_counts().index.tolist()
ingredients_keys = {ingredient: i for i, ingredient in enumerate(unique_ingredients)}
ingredients_keys

{'salt': 0,
 'olive oil': 1,
 'onions': 2,
 'water': 3,
 'garlic': 4,
 'sugar': 5,
 'garlic cloves': 6,
 'butter': 7,
 'ground black pepper': 8,
 'all-purpose flour': 9,
 'pepper': 10,
 'vegetable oil': 11,
 'eggs': 12,
 'soy sauce': 13,
 'kosher salt': 14,
 'green onions': 15,
 'tomatoes': 16,
 'large eggs': 17,
 'carrots': 18,
 'unsalted butter': 19,
 'ground cumin': 20,
 'extra-virgin olive oil': 21,
 'black pepper': 22,
 'milk': 23,
 'chili powder': 24,
 'oil': 25,
 'red bell pepper': 26,
 'purple onion': 27,
 'scallions': 28,
 'grated parmesan cheese': 29,
 'sesame oil': 30,
 'corn starch': 31,
 'ginger': 32,
 'baking powder': 33,
 'jalapeno chilies': 34,
 'dried oregano': 35,
 'chopped cilantro fresh': 36,
 'fresh lemon juice': 37,
 'diced tomatoes': 38,
 'fresh parsley': 39,
 'minced garlic': 40,
 'chicken broth': 41,
 'sour cream': 42,
 'cayenne pepper': 43,
 'fresh ginger': 44,
 'brown sugar': 45,
 'cooking spray': 46,
 'shallots': 47,
 'garlic powder': 48,
 'lime': 49,
 'lemo

In [25]:
def ingredients_to_vector(chosen_ingredients):
    vector = np.zeros(len(ingredients_keys))
    for ingredient in chosen_ingredients:
        index = ingredients_keys.get(ingredient)
        if ingredient in unique_ingredients:
            vector[index] = 1
    return vector

In [40]:
X = np.array([ingredients_to_vector(ings) for ings in recipe['ingredients']])
Y = recipe['cuisine']
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.]])

In [41]:
Y

0              greek
1        southern_us
2           filipino
3             indian
4             indian
            ...     
39769          irish
39770        italian
39771          irish
39772        chinese
39773        mexican
Name: cuisine, Length: 39774, dtype: object

In [67]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=40,stratify=Y)

In [68]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

base_rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

rf = CalibratedClassifierCV(base_rf, method='isotonic', cv=5)
rf.fit(X_train, Y_train)

CalibratedClassifierCV(cv=5,
                       estimator=RandomForestClassifier(n_estimators=300,
                                                        n_jobs=-1,
                                                        random_state=42),
                       method='isotonic')

In [136]:
probs = rf.predict_proba(X_test)
cuisines = rf.classes_

In [151]:
for i in range (5):
    order = np.argsort(probs[i])[::-1][:3]
    ingredient_names = np.array(unique_ingredients)
    present = np.array(ingredient_names[X_test[i] == 1])
    print("Ingredients:", list(present))
    print('\n')
    print("True:", Y_test.iloc[i])
    for j in order:
        print(f"({cuisines[j]}, {probs[i][j]*100:.2f}%)")
    print('\n')

Ingredients: ['onions', 'garlic', 'ground black pepper', 'all-purpose flour', 'dry white wine', 'white sugar', 'bay leaf', 'vegetable broth', 'french bread', 'brandy', 'ground thyme', 'fat-free parmesan cheese']


True: french
(french, 51.26%)
(italian, 28.68%)
(southern_us, 6.10%)


Ingredients: ['butter', 'all-purpose flour', 'baking powder', 'white sugar', 'ground ginger', 'confectioners sugar', 'golden syrup']


True: british
(british, 35.26%)
(southern_us, 23.79%)
(italian, 17.86%)


Ingredients: ['onions', 'garlic', 'oil', 'purple onion', 'chicken broth', 'cilantro', 'lime juice', 'pork tenderloin', 'shredded cabbage', 'ancho powder', 'hominy', 'red enchilada sauce', 'roasted tomatoes']


True: mexican
(mexican, 82.29%)
(thai, 3.97%)
(filipino, 3.60%)


Ingredients: ['salt', 'garlic cloves', 'extra-virgin olive oil', 'fresh parsley', 'bay leaf', 'ground red pepper', 'large shrimp', 'lemon wedge']


True: spanish
(italian, 48.85%)
(spanish, 29.26%)
(french, 13.76%)


Ingredients: 

In [138]:
y_true = np.asarray(Y_test)

top1 = 0
top3 = 0

for i in range(len(probs)):
    order = np.argsort(probs[i])[::-1]

    best = cuisines[order[0]]
    top_three = [cuisines[order[0]], cuisines[order[1]], cuisines[order[2]]]

    true_label = Y_test.iloc[i]

    if true_label == best:
        top1 += 1
    if true_label in top_three:
        top3 += 1

total = len(probs)
print(f"Top-1: {top1 / total * 100:.1f}%")
print(f"Top-3: {top3 / total * 100:.1f}%")

Top-1: 74.0%
Top-3: 90.4%


In [147]:
ing_count = X.sum(axis=0)
min_in_cuisine = 15

for cuisine in cuisines:
    rows = X[Y == cuisine]
    in_cuisine_count = rows.sum(axis=0)

    precision = in_cuisine_count / ing_count
    precision[in_cuisine_count < min_in_cuisine] = 0

    top10 = np.argsort(precision)[::-1][:10]

    print(cuisine.upper())
    for j in top10:
        print(f"  {ingredient_names[j]}: {in_cuisine_count[j]} / {ing_count[j]} ({precision[j] * 100:.1f}%)")
    print()

BRAZILIAN
  cachaca: 70.0 / 70.0 (100.0%)
  chocolate sprinkles: 16.0 / 18.0 (88.9%)
  dried black beans: 18.0 / 36.0 (50.0%)
  tapioca flour: 17.0 / 49.0 (34.7%)
  ice: 26.0 / 111.0 (23.4%)
  sweetened condensed milk: 57.0 / 271.0 (21.0%)
  ice cubes: 16.0 / 130.0 (12.3%)
  bananas: 21.0 / 224.0 (9.4%)
  coconut milk: 58.0 / 854.0 (6.8%)
  lime: 89.0 / 1439.0 (6.2%)

BRITISH
  stilton cheese: 17.0 / 18.0 (94.4%)
  suet: 16.0 / 18.0 (88.9%)
  golden syrup: 21.0 / 30.0 (70.0%)
  double cream: 15.0 / 40.0 (37.5%)
  malt vinegar: 18.0 / 51.0 (35.3%)
  puff pastry: 16.0 / 51.0 (31.4%)
  dried currants: 26.0 / 95.0 (27.4%)
  frozen pastry puff sheets: 25.0 / 106.0 (23.6%)
  plain flour: 36.0 / 154.0 (23.4%)
  caster sugar: 26.0 / 130.0 (20.0%)

CAJUN_CREOLE
  file powder: 46.0 / 49.0 (93.9%)
  andouille sausage: 224.0 / 267.0 (83.9%)
  creole seasoning: 216.0 / 276.0 (78.3%)
  cajun seasoning: 293.0 / 376.0 (77.9%)
  smoked sausage: 101.0 / 139.0 (72.7%)
  shrimp stock: 17.0 / 24.0 (70.8%)


array([1., 0., 0., ..., 0., 0., 0.])